> **Runtime: Google Colab required** — This notebook needs `tiktoken` and `transformers`, which require native C/Rust extensions unavailable in JupyterLite.
>
> [![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/L3GJ0N/course-notebooks-public/blob/main/02-tokenizer-playground.ipynb)

# Tokenizer Playground

**Lecture 4 — Tokenization: How Machines Read**

In this notebook you will explore the tokenizers used by real language models. You will see how text is split into tokens, compare tokenizers across models, discover the multilingual gap, and estimate token budgets for your own projects.

**Prerequisites:**
- Read Lecture 4 (Chapters 7-14) — you should understand subword tokenization and BPE
- A HuggingFace account (for loading model tokenizers)
- The [hf-llm-uv-starter](https://github.com/L3GJ0N/hf-llm-uv-starter) repo for environment setup

**What you will do:**
1. Tokenize text with tiktoken (GPT-4o's tokenizer)
2. Compare tokenizers across frontier models
3. Measure the multilingual token gap
4. Explore arithmetic tokenization failures
5. Dig into vocabularies — find patterns, glitch tokens, longest tokens
6. Estimate context window budgets for your projects

## 1. Setup

Install the required packages and import them. If you are using the [hf-llm-uv-starter](https://github.com/L3GJ0N/hf-llm-uv-starter), these should already be available.

In [ ]:
# Install packages (uncomment if needed)
# !pip install tiktoken transformers matplotlib

import tiktoken
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

## 2. First Contact: How GPT-4o Reads Text

Let's start with OpenAI's tokenizer. GPT-4o uses the `o200k_base` encoding — a byte-level BPE tokenizer with approximately 200,000 tokens in its vocabulary.

We'll write a small helper function that shows us exactly how a tokenizer splits text.

In [ ]:
# Load GPT-4o's tokenizer
enc = tiktoken.get_encoding("o200k_base")

def show_tokens(text: str, encoding=enc) -> None:
    """Tokenize text and display each token with its ID."""
    token_ids = encoding.encode(text)
    tokens = [encoding.decode([tid]) for tid in token_ids]
    
    print(f"Text:       {text!r}")
    print(f"Token IDs:  {token_ids}")
    print(f"Tokens:     {tokens}")
    print(f"Count:      {len(token_ids)} tokens")
    print()

# Try it!
show_tokens("strawberry")

### Your Turn: The Strawberry Test and Beyond

Tokenize each of the following. For each one, note: how many tokens? Where do the splits fall? Any surprises?

In [ ]:
# TODO: Tokenize each of these and observe the splits
words = [
    "strawberry",
    "mississippi",
    "unbreakable",
    # Add your own first name here:
    # "YourName",
    "Hochschule Aalen",
    "Stanford University",
    "def calculate_average(values: list[float]) -> float:",
    'public static double calculateAverage(List<Double> values) {',
]

for word in words:
    show_tokens(word)

**Questions to answer:**
- How many tokens does `Hochschule Aalen` take compared to `Stanford University`? Why?
- Does `unbreakable` split into `un` + `break` + `able`? Or something different?
- Compare the Python and Java function signatures — which uses more tokens? Why might that matter for code generation?

## 3. Compare Tokenizers Across Models

Different models use different tokenizers — trained on different data, with different vocabulary sizes. Let's load several and compare them side by side.

| Model | Tokenizer | Vocab Size | Library |
|-------|-----------|------------|---------|
| GPT-4o | o200k_base | ~200K | tiktoken |
| GPT-4 | cl100k_base | ~100K | tiktoken |
| LLaMA 3 | BPE | 128K | HuggingFace |
| Gemma 2 | SentencePiece | 256K | HuggingFace |

In [ ]:
# Load multiple tokenizers
tokenizers = {
    "GPT-4o":  ("tiktoken", tiktoken.get_encoding("o200k_base")),
    "GPT-4":   ("tiktoken", tiktoken.get_encoding("cl100k_base")),
    "LLaMA 3": ("hf", AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")),
    "Gemma 2":  ("hf", AutoTokenizer.from_pretrained("google/gemma-2-2b")),
}

def count_tokens(text: str, name: str, kind: str, tok) -> int:
    """Count tokens for a given text using the specified tokenizer."""
    if kind == "tiktoken":
        return len(tok.encode(text))
    else:  # HuggingFace
        return len(tok.encode(text))

def compare_tokenizers(text: str) -> None:
    """Show token counts across all loaded tokenizers."""
    print(f"Text: {text!r}")
    print("-" * 50)
    for name, (kind, tok) in tokenizers.items():
        n = count_tokens(text, name, kind, tok)
        print(f"  {name:12s}: {n:4d} tokens")
    print()

In [ ]:
# Compare on several test strings
test_strings = [
    "Hello, world!",
    "def calculate_average(values: list[float]) -> float:",
    "Künstliche Intelligenz verändert die Softwareentwicklung.",
    "The quick brown fox jumps over the lazy dog.",
    "1234 + 5678 = 6912",
]

for text in test_strings:
    compare_tokenizers(text)

**Questions to answer:**
- Which tokenizer produces the fewest tokens overall? Why might that be?
- Do all tokenizers split the German sentence the same way?
- Do you see any case where a larger vocabulary actually produces *more* tokens?

## 4. The Multilingual Gap

In Lecture 4 (Chapter 17) we learned that non-English text costs more tokens. Let's measure this precisely.

**Task:** Tokenize the same sentence in English and German (and optionally a third language). Calculate the token ratio.

In [ ]:
sentences = {
    "English": "Artificial intelligence is changing software development.",
    "German":  "Künstliche Intelligenz verändert die Softwareentwicklung.",
    # TODO: Add a third language if you speak one
    # "Spanish": "La inteligencia artificial está cambiando el desarrollo de software.",
    # "Chinese": "人工智能正在改变软件开发。",
}

# Count tokens per language for each tokenizer
results = {}
for lang, text in sentences.items():
    results[lang] = {}
    for name, (kind, tok) in tokenizers.items():
        results[lang][name] = count_tokens(text, name, kind, tok)

# Display as a table
print(f"{'Language':12s}", end="")
for name in tokenizers:
    print(f"  {name:>10s}", end="")
print()
print("-" * (12 + 12 * len(tokenizers)))
for lang, counts in results.items():
    print(f"{lang:12s}", end="")
    for name in tokenizers:
        print(f"  {counts[name]:>10d}", end="")
    print()

# Calculate ratios relative to English
print("\nRatio (relative to English):")
for lang, counts in results.items():
    if lang == "English":
        continue
    print(f"  {lang}:", end="")
    for name in tokenizers:
        ratio = counts[name] / results["English"][name]
        print(f"  {name}: {ratio:.2f}x", end="")
    print()

### Visualize the Gap

Let's make a bar chart to see the multilingual token cost at a glance.

In [ ]:
# Bar chart: token counts per language, grouped by tokenizer
fig, ax = plt.subplots(figsize=(10, 5))

languages = list(results.keys())
tok_names = list(tokenizers.keys())
x = range(len(languages))
width = 0.8 / len(tok_names)

for i, name in enumerate(tok_names):
    counts = [results[lang][name] for lang in languages]
    offset = (i - len(tok_names) / 2 + 0.5) * width
    ax.bar([xi + offset for xi in x], counts, width, label=name)

ax.set_xlabel("Language")
ax.set_ylabel("Token Count")
ax.set_title("Same Sentence, Different Languages — Token Cost")
ax.set_xticks(x)
ax.set_xticklabels(languages)
ax.legend()
plt.tight_layout()
plt.show()

## 5. The Arithmetic Breakdown

In Chapter 7 we saw that models struggle with arithmetic. The root cause: digits get split inconsistently across token boundaries. Let's see this directly.

In [ ]:
# Tokenize numbers and see where the boundaries fall
numbers = ["999", "1000", "1001", "1234", "5678", "2249", "2250", "2251"]

print("How GPT-4o tokenizes numbers:")
print("-" * 50)
for num in numbers:
    token_ids = enc.encode(num)
    tokens = [enc.decode([tid]) for tid in token_ids]
    print(f"  {num:>6s}  →  {tokens}  ({len(tokens)} token{'s' if len(tokens) > 1 else ''})")

print()
print("Notice: three consecutive numbers (2249, 2250, 2251)")
print("get completely different internal representations!")
print()

# Now tokenize an arithmetic expression
show_tokens("1234 + 5678")
show_tokens("3.14159")

**Questions to answer:**
- Can you see why 1234 + 5678 would be hard for a model? Where do the place-value columns fall relative to token boundaries?
- Do ALL tokenizers split these numbers the same way? Try with the other tokenizers from Section 3.

## 6. How Code Gets Tokenized

This is a course about AI-supported **software development**. Code has its own tokenization properties that directly affect how well Copilot, Claude Code, and other AI tools work. Let's explore them.

In Lecture 4 (Chapter 19), we discuss five code-specific tokenization phenomena: indentation cost, naming conventions, boilerplate efficiency, programming language differences, and comments as context. Here you will verify each one empirically.

In [ ]:
# --- Experiment 1: Indentation cost ---
# Same logic, different nesting depth

flat_code = """def process(items):
    results = []
    for item in items:
        if item.is_valid():
            results.append(item.value)
    return results"""

nested_code = """def process(data):
    results = []
    for category in data:
        for group in category.groups:
            for item in group.items:
                if item.is_valid():
                    if item.value > 0:
                        results.append(item.value)
    return results"""

print("=== Indentation Cost ===")
show_tokens(flat_code)
show_tokens(nested_code)
print(f"Flat code:   {len(enc.encode(flat_code)):3d} tokens")
print(f"Nested code: {len(enc.encode(nested_code)):3d} tokens")
print(f"Difference:  {len(enc.encode(nested_code)) - len(enc.encode(flat_code)):+3d} tokens (deeper nesting costs more)")

In [ ]:
# --- Experiment 2: Naming conventions ---
# Same identifier, different styles

identifiers = {
    "camelCase":    "calculateAverageScore",
    "snake_case":   "calculate_average_score",
    "PascalCase":   "CalculateAverageScore",
    "SCREAMING":    "CALCULATE_AVERAGE_SCORE",
    "short":        "calc_avg",
    "single_char":  "x",
}

print("=== Naming Convention Token Cost ===")
print(f"{'Style':15s}  {'Identifier':30s}  {'Tokens':>6s}")
print("-" * 55)
for style, name in identifiers.items():
    n = len(enc.encode(name))
    tokens = [enc.decode([t]) for t in enc.encode(name)]
    print(f"{style:15s}  {name:30s}  {n:>6d}  → {tokens}")

In [ ]:
# --- Experiment 3: Boilerplate vs novel code ---

boilerplate_snippets = [
    ("__init__",       "def __init__(self):"),
    ("__str__",        "def __str__(self):"),
    ("import",         "from collections import defaultdict"),
    ("main guard",     'if __name__ == "__main__":'),
]

novel_snippets = [
    ("domain func",    "def calibrate_magnetometer(readings: list[SensorDatum]):"),
    ("custom class",   "class QuantumStateEstimator(BaseEstimator):"),
    ("domain import",  "from neuromorphic_simulator.core import SpikeTrainAnalyzer"),
    ("business rule",  'if customer.loyalty_tier == "PLATINUM" and order.subtotal > 500:'),
]

print("=== Boilerplate vs Novel Code ===")
print(f"\n{'Type':15s}  {'Label':15s}  {'Tokens':>6s}  Code")
print("-" * 80)
for label, code in boilerplate_snippets:
    n = len(enc.encode(code))
    print(f"{'Boilerplate':15s}  {label:15s}  {n:>6d}  {code}")
for label, code in novel_snippets:
    n = len(enc.encode(code))
    print(f"{'Novel':15s}  {label:15s}  {n:>6d}  {code}")

In [ ]:
# --- Experiment 4: Same algorithm, different programming languages ---

python_code = '''def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)'''

java_code = '''public static int fibonacci(int n) {
    if (n <= 1) {
        return n;
    }
    return fibonacci(n - 1) + fibonacci(n - 2);
}'''

rust_code = '''fn fibonacci(n: u32) -> u32 {
    if n <= 1 {
        return n;
    }
    fibonacci(n - 1) + fibonacci(n - 2)
}'''

print("=== Same Algorithm, Different Languages ===")
for lang, code in [("Python", python_code), ("Java", java_code), ("Rust", rust_code)]:
    n = len(enc.encode(code))
    chars = len(code)
    print(f"  {lang:8s}: {n:3d} tokens, {chars:3d} chars, ratio: {chars/n:.1f} chars/token")

In [ ]:
# --- Experiment 5: Comments as context cost ---

code_no_comments = '''def validate_email(address):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, address))'''

code_with_comments = '''def validate_email(address):
    # RFC 5322 compliant email validation
    # Checks: local part (alphanumeric + special chars), @ symbol,
    # domain (alphanumeric + hyphens), TLD (2+ letters)
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, address))'''

n_no = len(enc.encode(code_no_comments))
n_with = len(enc.encode(code_with_comments))

print("=== Comments as Context Cost ===")
print(f"  Without comments: {n_no:3d} tokens")
print(f"  With comments:    {n_with:3d} tokens")
print(f"  Comment cost:     {n_with - n_no:+3d} tokens ({(n_with - n_no) / n_no * 100:.0f}% overhead)")
print(f"\n  Those {n_with - n_no} extra tokens tell the model WHY this code exists.")
print(f"  Is that worth it? Depends on the task.")

**Questions to answer:**
- Which naming convention is most token-efficient? Does that surprise you?
- How many extra tokens does deep nesting cost? What would you do to reduce it?
- Java vs Python: how much more "context budget" does the model have when generating Python?
- When are comments worth their token cost? When are they wasteful?
- If you were writing a CLAUDE.md rule about code style, what would you recommend based on these findings?

## 7. Vocabulary Archaeology

Every tokenizer has a vocabulary — a fixed list of tokens it knows. Let's dig into GPT-4o's vocabulary and see what's inside.

In [ ]:
# How big is the vocabulary?
print(f"GPT-4o vocabulary size: {enc.n_vocab:,} tokens")

# Decode the first 200 tokens — what do they look like?
print("\nFirst 200 tokens in the vocabulary:")
print("-" * 60)
for i in range(200):
    try:
        token_str = enc.decode([i])
        print(f"  {i:5d}: {token_str!r}")
    except Exception:
        print(f"  {i:5d}: [cannot decode]")

### Find the Longest Token

What is the longest single token in the vocabulary? This tells you about the most common phrases in the training data.

In [ ]:
# TODO: Find the longest token in the vocabulary
# Hint: decode each token ID, measure its length, track the maximum
#
# Strategy: iterate over a range of token IDs, decode each one,
# and keep track of the longest decoded string.
# Note: some token IDs may fail to decode — wrap in try/except.

longest_token = ""
longest_id = -1

for i in range(enc.n_vocab):
    try:
        token_str = enc.decode([i])
        if len(token_str) > len(longest_token):
            longest_token = token_str
            longest_id = i
    except Exception:
        continue

print(f"Longest token: ID {longest_id}")
print(f"Content: {longest_token!r}")
print(f"Length: {len(longest_token)} characters")

### Glitch Token Exploration

In Chapter 18 of the lecture, we discussed glitch tokens — tokens that exist in the vocabulary but cause the model to malfunction because they were never properly trained. Let's check if some famous ones still exist.

In [ ]:
# Check famous glitch tokens — do they tokenize as single tokens?
glitch_candidates = [
    " SolidGoldMagikarp",  # note the leading space
    " TheNitromeFan",
    "SolidGoldMagikarp",   # without leading space
]

print("Glitch token check (GPT-4o / o200k_base):")
print("-" * 50)
for text in glitch_candidates:
    token_ids = enc.encode(text)
    tokens = [enc.decode([tid]) for tid in token_ids]
    is_single = len(token_ids) == 1
    status = "SINGLE TOKEN (potential glitch!)" if is_single else f"split into {len(token_ids)} tokens"
    print(f"  {text!r:30s} → {status}")
    print(f"    Tokens: {tokens}")
    print()

# Also check with the older GPT-4 tokenizer (cl100k_base)
enc_old = tiktoken.get_encoding("cl100k_base")
print("Same check with GPT-4 / cl100k_base:")
print("-" * 50)
for text in glitch_candidates:
    token_ids = enc_old.encode(text)
    tokens = [enc_old.decode([tid]) for tid in token_ids]
    is_single = len(token_ids) == 1
    status = "SINGLE TOKEN" if is_single else f"split into {len(token_ids)} tokens"
    print(f"  {text!r:30s} → {status}")

## 8. Token Budget Estimation

Everything in the context window costs tokens — your prompt, CLAUDE.md, the model's response, tool results. Let's calculate a real budget.

**Task:** Tokenize your CLAUDE.md from the Locust exercise (Lecture 3) and estimate how much of the context window it consumes.

In [ ]:
# TODO: Paste your CLAUDE.md content here (or load from file)
claude_md = """
# Your CLAUDE.md content here
# Paste the content from your Locust exercise
"""

# Tokenize it
claude_md_tokens = len(enc.encode(claude_md))
print(f"Your CLAUDE.md: {claude_md_tokens:,} tokens")

# Context budget calculation
context_window = 200_000  # Claude's context window
system_prompt_estimate = 3_000  # approximate system prompt size
typical_prompt = 200  # a typical user prompt

remaining = context_window - system_prompt_estimate - claude_md_tokens - typical_prompt

print(f"\n--- Context Budget (Claude, 200K window) ---")
print(f"  Context window:    {context_window:>10,} tokens")
print(f"  System prompt:    -{system_prompt_estimate:>10,} tokens (estimate)")
print(f"  Your CLAUDE.md:   -{claude_md_tokens:>10,} tokens")
print(f"  Typical prompt:   -{typical_prompt:>10,} tokens")
print(f"                    {'─' * 20}")
print(f"  Remaining:         {remaining:>10,} tokens ({remaining/context_window*100:.1f}%)")
print()

# What if CLAUDE.md grew?
for size_label, size in [("1,000 tokens", 1000), ("5,000 tokens", 5000), ("10,000 tokens", 10000)]:
    rem = context_window - system_prompt_estimate - size - typical_prompt
    print(f"  If CLAUDE.md = {size_label}: {rem:,} remaining ({rem/context_window*100:.1f}%)")

## 9. Deliverables

Document your findings in an **AI Diary entry** with:

1. At least **5 surprising tokenization results** with your explanations of why they happen
2. The **English-German token count comparison** with your calculated ratio
3. Your **CLAUDE.md token budget estimate** — how much of the context window does it consume?
4. **One thing you would change** about your prompting approach based on what you now understand about tokenization
5. **One code style recommendation** based on your code tokenization experiments (Section 6)

### Reflection Questions

- Which tokenizer was most efficient for German text? Why?
- If you were building a model for a German-speaking audience, what would you change about the tokenizer?
- How does knowing about token budgets change how you write CLAUDE.md files?
- Should you optimize your code for token efficiency? When does it matter, when doesn't it?